# 🎬 Old Video Enhancer
**SeedVR2 (AI restoration) → shape fix → color → smooth motion → save**

1. **Runtime → Change runtime type → A100 GPU** (or L4).
2. Set the switches in each section, then **Runtime → Run all**.
3. Start with **Test mode ON** (a few seconds). When you like the result, turn it off for the whole video.
4. When done: run **5.3 Shut down**, or turn it on before a long run so Colab stops by itself.

## 1 · Setup

In [ ]:
#@title 1.1 Check GPU
import torch
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name}  ({vram:.0f} GB)')
BIG_GPU = vram >= 30          # A100 40/80GB
if vram < 20:
    print('WARNING: this GPU is too small for SeedVR2. Switch to A100 or L4.')
print('Mode:', 'A100 (full speed)' if BIG_GPU else 'L4 (memory-saving mode)')

In [ ]:
#@title 1.2 Install SeedVR2 (2-3 minutes)
%cd /content
!rm -rf /content/seedvr2
!git clone --depth 1 -q https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git /content/seedvr2
!grep -vE '^(torch|torchvision|torchaudio)$' /content/seedvr2/requirements.txt > /content/req_colab.txt
!pip install -q -r /content/req_colab.txt ccvfi
!python /content/seedvr2/inference_cli.py --help > /dev/null && echo 'Install done'

## 2 · Input

In [ ]:
#@title 2.1 Load the video
SOURCE = 'Google Drive' #@param ['Google Drive', 'Upload from computer', 'Already in Colab']
VIDEO_PATH = '/content/drive/MyDrive/my_old_video.mp4' #@param {type:"string"}
#@markdown **Google Drive**: full path in Drive · **Already in Colab**: a file in the Files panel, e.g. `/content/my_old_video.mp4` · **Upload**: path is ignored.

import os, shutil
from google.colab import drive, files
os.makedirs('/content/input', exist_ok=True)

if SOURCE == 'Google Drive':
    drive.mount('/content/drive')
    path = VIDEO_PATH
elif SOURCE == 'Upload from computer':
    up = files.upload()
    path = '/content/' + list(up.keys())[0]
else:
    path = VIDEO_PATH

VIDEO_NAME = os.path.splitext(os.path.basename(path))[0]
SRC = '/content/input/original' + os.path.splitext(path)[1]
shutil.copy(path, SRC)
print('Video ready:', SRC)
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,display_aspect_ratio:format=duration -of default=nw=1 "$SRC"

In [ ]:
#@title 2.2 Choose the part to process
TEST_MODE = True #@param {type:"boolean"}
#@markdown ✅ on = short test clip · ☐ off = **whole video**
TEST_START = 30 #@param {type:"integer"}
TEST_SECONDS = 5 #@param {type:"slider", min:2, max:60, step:1}
DEINTERLACE = False #@param {type:"boolean"}
#@markdown Deinterlace only if you see horizontal comb lines on movement.

vf = '-vf yadif' if DEINTERLACE else ''
trim = f'-ss {TEST_START} -t {TEST_SECONDS}' if TEST_MODE else ''
CLIP = '/content/clip.mp4'
!ffmpeg -y -loglevel error $trim -i "$SRC" $vf -c:v libx264 -crf 14 -pix_fmt yuv420p -c:a aac "$CLIP"
!ffprobe -v error -count_frames -select_streams v:0 -show_entries stream=nb_read_frames -of default=nw=1 "$CLIP"
print('Processing:', f'{TEST_SECONDS}s test from {TEST_START}s' if TEST_MODE else 'the WHOLE video')

## 3 · AI restoration

In [ ]:
#@title 3.1 SeedVR2 (main AI step)
RUN_SEEDVR2 = True #@param {type:"boolean"}
MODEL = '3b' #@param ['3b', '7b']
#@markdown **3b** = faithful to the original, faster · **7b** = more detail, invents more, slower (A100 only)
RESOLUTION = 1080 #@param [720, 1080, 1440] {type:"raw"}
BATCH = 33 #@param [9, 17, 21, 33, 49, 65] {type:"raw"}
#@markdown Batch = frames processed together. Higher = steadier picture, more GPU memory.
CHUNK = 330 #@param [0, 165, 330, 660] {type:"raw"}
#@markdown Frames processed per chunk. Saving needs several copies of every frame in RAM (~45 MB per 1080p frame at peak), so 1,800 frames at once crashed an 83 GB A100 runtime. **330 ≈ 15 GB, safe for any length** · 660 = fewer seams, ~30 GB · 0 = all at once, only for clips under ~10 s.
VAE_TILE = 1536 #@param [0, 1024, 1536] {type:"raw"}
#@markdown Measured on A100 40GB: 1536 is only ~5% faster than 1024 and already uses ~34 GB GPU RAM · **0** = no tiling, likely out of memory on 40 GB · out of memory → **1024**
COLOR_MATCH = 'lab' #@param ['lab', 'wavelet', 'wavelet_adaptive', 'hsv', 'adain', 'none']
#@markdown Keeps the AI result's colors matched to the original.
SEED = 42 #@param {type:"integer"}

import glob, time
if not RUN_SEEDVR2:
    RAW = CLIP
    print('SeedVR2 skipped - the next steps use the original clip.')
else:
    if BIG_GPU:
        dit = 'seedvr2_ema_7b_fp16.safetensors' if MODEL == '7b' else 'seedvr2_ema_3b_fp16.safetensors'
        mem = '' if VAE_TILE == 0 else (f'--vae_encode_tiled --vae_decode_tiled '
                                        f'--vae_encode_tile_size {VAE_TILE} --vae_decode_tile_size {VAE_TILE}')
    else:  # L4: smaller model file + offload to save memory
        dit = 'seedvr2_ema_3b_fp8_e4m3fn.safetensors'
        mem = ('--vae_encode_tiled --vae_decode_tiled --dit_offload_device cpu '
               '--vae_offload_device cpu --blocks_to_swap 16')
        BATCH = min(BATCH, 21)

    OUT_DIR = '/content/seedvr_out'
    !rm -rf $OUT_DIR && mkdir -p $OUT_DIR
    t0 = time.time()
    cmd = (f'python /content/seedvr2/inference_cli.py "{CLIP}" --output {OUT_DIR}/ '
           f'--model_dir /content/models/SEEDVR2 --dit_model {dit} --resolution {RESOLUTION} '
           f'--batch_size {BATCH} --uniform_batch_size --temporal_overlap 3 --prepend_frames 4 '
           f'--color_correction {COLOR_MATCH} --seed {SEED} --video_backend ffmpeg '
           f'--chunk_size {CHUNK} {mem}')
    !{cmd}
    print(f'Took {(time.time() - t0) / 60:.1f} minutes')

    found = sorted(glob.glob(f'{OUT_DIR}/**/*.mp4', recursive=True) +
                   glob.glob('/content/seedvr2/output/**/*.mp4', recursive=True), key=os.path.getmtime)
    assert found, 'No output video found - scroll up for the error message'
    RAW = found[-1]
    print('Result:', RAW)

## 4 · Finishing

In [ ]:
#@title 4.1 Audio + picture shape
FIX_ASPECT = True #@param {type:"boolean"}
#@markdown Old 720×480 video often has non-square pixels (shown as 4:3). Without this, people can look ~12% too wide.

import json, subprocess
def probe(p):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
                        'stream=width,height,sample_aspect_ratio,display_aspect_ratio',
                        '-of', 'json', p], capture_output=True, text=True)
    return json.loads(r.stdout)['streams'][0]

src_info, out_info = probe(SRC), probe(RAW)
print('Original:', src_info)
ow, oh = out_info['width'], out_info['height']
tw = ow
dar = src_info.get('display_aspect_ratio', '')
if FIX_ASPECT and ':' in dar:
    num, den = (int(v) for v in dar.split(':'))
    if num > 0 and den > 0:
        tw = round(oh * num / den / 2) * 2

FINAL = '/content/step1_restored.mp4'
if tw != ow:
    print(f'Fixing shape: {ow}x{oh} -> {tw}x{oh}')
    !ffmpeg -y -loglevel error -i "$RAW" -i "$CLIP" -map 0:v -map 1:a? -vf "scale={tw}:{oh}:flags=lanczos,setsar=1" -c:v libx264 -crf 16 -pix_fmt yuv420p -c:a aac -shortest "$FINAL"
else:
    print(f'Shape is already correct: {ow}x{oh}')
    !ffmpeg -y -loglevel error -i "$RAW" -i "$CLIP" -map 0:v -map 1:a? -c:v copy -c:a aac -shortest "$FINAL"
LATEST = FINAL

In [ ]:
#@title 4.2 Color correction
COLOR_FIX = True #@param {type:"boolean"}
WARM_FIX = 0.1 #@param {type:"slider", min:0, max:0.3, step:0.02}
#@markdown Removes the orange cast of the stage lights. 0 = none · 0.10 = gentle · 0.20+ = strong
SATURATION = 0.88 #@param {type:"slider", min:0.6, max:1.2, step:0.02}
BRIGHTNESS = 0.0 #@param {type:"slider", min:-0.1, max:0.1, step:0.01}
CONTRAST = 1.0 #@param {type:"slider", min:0.8, max:1.2, step:0.02}

LATEST = FINAL
if COLOR_FIX:
    COLOR = '/content/step2_color.mp4'
    w = WARM_FIX
    vf = (f'colorbalance=rm={-0.8 * w:.3f}:bm={w:.3f}:rh={-w:.3f}:bh={1.2 * w:.3f},'
          f'eq=saturation={SATURATION}:brightness={BRIGHTNESS}:contrast={CONTRAST}')
    !ffmpeg -y -loglevel error -i "$FINAL" -vf "$vf" -c:v libx264 -crf 16 -pix_fmt yuv420p -c:a copy "$COLOR"
    LATEST = COLOR
    print('Color done')
else:
    print('Color correction skipped')

In [ ]:
#@title 4.3 Smooth motion (RIFE, doubles the frame rate)
SMOOTH_FPS = True #@param {type:"boolean"}
SCENE_CUT = 35 #@param {type:"slider", min:15, max:60, step:1}
#@markdown Camera cuts are not blended. Ghosting at cuts → **lower** · jumps in fast motion → **higher**

if SMOOTH_FPS:
    import cv2, subprocess, numpy as np
    from ccvfi import AutoModel, ConfigType
    rife = AutoModel.from_pretrained(pretrained_model_name=ConfigType.RIFE_IFNet_v426_heavy)

    SRC_V = LATEST
    SMOOTH = '/content/step3_smooth.mp4'
    cap = cv2.VideoCapture(SRC_V)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'{w}x{h}  {fps:.2f} fps -> {fps * 2:.2f} fps,  {total} frames')

    enc = subprocess.Popen(
        ['ffmpeg', '-y', '-loglevel', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24',
         '-s', f'{w}x{h}', '-r', f'{fps * 2:.3f}', '-i', '-', '-i', SRC_V,
         '-map', '0:v', '-map', '1:a?', '-c:v', 'libx264', '-crf', '16', '-pix_fmt', 'yuv420p',
         '-c:a', 'copy', '-shortest', SMOOTH], stdin=subprocess.PIPE)

    small = lambda f: cv2.cvtColor(cv2.resize(f, (160, 120)), cv2.COLOR_BGR2GRAY).astype(np.int16)
    ok, prev = cap.read()
    n, cuts = 0, 0
    while ok:
        ok, cur = cap.read()
        enc.stdin.write(prev.tobytes())
        if not ok:
            enc.stdin.write(prev.tobytes())          # keep the frame count even at the end
            break
        if np.abs(small(prev) - small(cur)).mean() > SCENE_CUT:
            mid = prev; cuts += 1                    # camera cut: don't blend two different shots
        else:
            mid = rife.inference_image_list(img_list=[prev, cur])[0]
        enc.stdin.write(np.ascontiguousarray(mid).tobytes())
        prev = cur
        n += 1
        if n % 200 == 0:
            print(f'{n}/{total} frames')
    cap.release(); enc.stdin.close(); enc.wait()
    LATEST = SMOOTH
    print(f'Done - {cuts} camera cuts detected.')
else:
    print('Smooth motion skipped')

## 5 · Review & save

In [ ]:
#@title 5.1 Compare: original (left) vs final (right)
T = 2 #@param {type:"slider", min:0, max:60, step:0.5}
#@markdown Second inside the processed part. In test mode, the original video time is **Test start + this**.
ZOOM_X = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
ZOOM_Y = 0.35 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
#@markdown Zoom center: left→right, top→bottom. Put it on a face.

import subprocess
from PIL import Image as PImage
from IPython.display import Image, display
dur = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                            '-of', 'default=nw=1:nk=1', CLIP], capture_output=True, text=True).stdout)
t = min(T, max(dur - 0.5, 0))
!ffmpeg -y -loglevel error -ss $t -i "$CLIP" -frames:v 1 /content/s_orig.png
!ffmpeg -y -loglevel error -ss $t -i "$LATEST" -frames:v 1 /content/s_enh.png
e = PImage.open('/content/s_enh.png').convert('RGB')
o = PImage.open('/content/s_orig.png').convert('RGB').resize(e.size, PImage.BICUBIC)
W, H = e.size
both = PImage.new('RGB', (W * 2, H)); both.paste(o, (0, 0)); both.paste(e, (W, 0))
both.save('/content/s_compare.png')
s = W // 5
x, y = int(W * ZOOM_X) - s // 2, int(H * ZOOM_Y) - s // 2
crop = lambda im: im.crop((x, y, x + s, y + s)).resize((s * 3, s * 3), PImage.NEAREST)
z = PImage.new('RGB', (s * 6, s * 3)); z.paste(crop(o), (0, 0)); z.paste(crop(e), (s * 3, 0))
z.save('/content/s_zoom.png')
print(f'Frame at {t}s of the processed part = {t + (TEST_START if TEST_MODE else 0)}s in the original video')
display(Image('/content/s_compare.png', width=1600))
display(Image('/content/s_zoom.png', width=1600))

In [ ]:
#@title 5.2 Save the result
SAVE_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER = '/content/drive/MyDrive/video-enhancer' #@param {type:"string"}
DOWNLOAD_TO_COMPUTER = False #@param {type:"boolean"}
OUTPUT_NAME = '' #@param {type:"string"}
#@markdown Leave the name empty for an automatic one, e.g. `my_old_video_3b_1080p_test.mp4`

name = OUTPUT_NAME or f"{VIDEO_NAME}_{MODEL if RUN_SEEDVR2 else 'orig'}_{RESOLUTION}p_{'test' if TEST_MODE else 'full'}.mp4"
if not name.endswith('.mp4'):
    name += '.mp4'
out_file = '/content/' + name
shutil.copy(LATEST, out_file)

if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    shutil.copy(out_file, os.path.join(DRIVE_FOLDER, name))
    print('Saved to Drive:', os.path.join(DRIVE_FOLDER, name))
if DOWNLOAD_TO_COMPUTER:
    files.download(out_file)

In [ ]:
#@title 5.3 Shut down (stop using compute units)
SHUT_DOWN = False #@param {type:"boolean"}
#@markdown Turn on **after** the file is saved, or before a long *Run all* so Colab stops by itself at the end.
#@markdown ⚠️ Deletes everything in this Colab session (your Drive copy is safe).

if SHUT_DOWN:
    from google.colab import runtime
    print('Shutting down the runtime...')
    runtime.unassign()
else:
    print('Runtime still on - remember: Runtime → Disconnect and delete runtime')